# Random Forest Demand Predictions

Train a simple Random Forest on the cleaned training data and predict `num_orders` for every row in `test.csv` (each row represents a center, meal, and week). A chronological holdout checks prediction error before the final model is fitted on all training weeks. The source CSV files are not modified.

In [2]:
# Import tools for loading data, training the model, and measuring validation error.
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Find the project root whether this notebook is run from the root or notebooks folder.
project_root = Path.cwd()
if not (project_root / "data" / "raw" / "cleaned_merged_data.csv").exists():
    project_root = project_root.parent

# Read the cleaned labeled data and the unlabeled test rows without changing either file.
train_data = pd.read_csv(project_root / "data" / "raw" / "train.csv")
test_data = pd.read_csv(project_root / "data" / "raw" / "test.csv")

print("Training rows and columns:", train_data.shape)
print("Test rows and columns:", test_data.shape)
print("Training week range:", train_data["week"].min(), "to", train_data["week"].max())
print("Test week range:", test_data["week"].min(), "to", test_data["week"].max())

Training rows and columns: (456548, 9)
Test rows and columns: (32573, 8)
Training week range: 1 to 145
Test week range: 146 to 155


In [3]:
# Use only fields present in both files; identifiers are categorical, not numeric quantities.
feature_columns = [
    "week",
    "center_id",
    "meal_id",
    "checkout_price",
    "base_price",
    "emailer_for_promotion",
    "homepage_featured",
]
categorical_columns = ["center_id", "meal_id"]

# Remove only rows missing a required feature or the training target, and report each row count.
training_rows_before = len(train_data)
test_rows_before = len(test_data)
train_data = train_data.dropna(subset=feature_columns + ["num_orders"]).reset_index(drop=True)
test_data = test_data.dropna(subset=feature_columns).reset_index(drop=True)
print("Training rows after required-field drop:", len(train_data), "(removed", training_rows_before - len(train_data), ")")
print("Test rows after required-field drop:", len(test_data), "(removed", test_rows_before - len(test_data), ")")

# Convert IDs to text so one-hot encoding does not treat them as ordered numbers.
train_features_raw = train_data[feature_columns].copy()
test_features_raw = test_data[feature_columns].copy()
for column in categorical_columns:
    train_features_raw[column] = train_features_raw[column].astype(str)
    test_features_raw[column] = test_features_raw[column].astype(str)

# Encode both sets together to guarantee the training and test columns match.
combined_features = pd.concat([train_features_raw, test_features_raw], ignore_index=True)
combined_features = pd.get_dummies(combined_features, columns=categorical_columns, dtype=int)
train_features = combined_features.iloc[: len(train_data)].reset_index(drop=True)
test_features = combined_features.iloc[len(train_data) :].reset_index(drop=True)
print("Encoded training features:", train_features.shape)
print("Encoded test features:", test_features.shape)

Training rows after required-field drop: 456548 (removed 0 )
Test rows after required-field drop: 32573 (removed 0 )
Encoded training features: (456548, 133)
Encoded test features: (32573, 133)


In [4]:
# Hold out the latest 10 training weeks to check performance on later weeks.
validation_start_week = int(train_data["week"].max()) - 9
fit_mask = train_data["week"] < validation_start_week
validation_mask = train_data["week"] >= validation_start_week

# Fit a modest random forest; each tree learns demand patterns from a bootstrap sample.
validation_model = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)
validation_model.fit(train_features.loc[fit_mask], train_data.loc[fit_mask, "num_orders"])

# Predict the held-out weeks and report average absolute and root-mean-squared errors in orders.
validation_predictions = validation_model.predict(train_features.loc[validation_mask])
validation_actual = train_data.loc[validation_mask, "num_orders"]
mae = mean_absolute_error(validation_actual, validation_predictions)
rmse = np.sqrt(mean_squared_error(validation_actual, validation_predictions))
print("Validation weeks:", validation_start_week, "to", int(train_data["week"].max()))
print("Validation rows:", int(validation_mask.sum()))
print(f"Mean absolute error (orders): {mae:,.2f}")
print(f"Root mean squared error (orders): {rmse:,.2f}")

KeyboardInterrupt: 

In [ ]:
# Refit the same simple model using every cleaned training row for final predictions.
random_forest_model = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)
random_forest_model.fit(train_features, train_data["num_orders"])

# Predict one order count per test row and prevent impossible negative count predictions.
predicted_orders = np.maximum(random_forest_model.predict(test_features), 0)
predictions = test_data[["id", "week"]].copy()
predictions["predicted_num_orders"] = predicted_orders

# Save this derived output outside the raw-data folder and show a few predictions.
output_directory = project_root / "data" / "processed"
output_directory.mkdir(parents=True, exist_ok=True)
output_path = output_directory / "random_forest_predictions.csv"
predictions.to_csv(output_path, index=False)
print("Predicted rows:", len(predictions))
print("Prediction file:", output_path)
predictions.head()